![Dwengo](images/dwengo.png) 

# Reconnaître les émotions

Dans cette activité, tu vas développer un système d'IA capable de reconnaître les émotions. Cela te permettra d'apprendre, étape par étape, différents principes de l'intelligence artificielle et de l'apprentissage automatique.


## Connaissances préalables

Pour commencer avec ce notebook, tu as besoin de connaissances de base en programmation Python. Dans ce notebook, tu utiliseras des types de données, des opérateurs, des structures et des fonctions. Si tu n’es pas sûr d’avoir suffisamment de connaissances en Python pour ce notebook, tu peux consulter [dwengo.org/python](https://dwengo.org/python). Les principes de base y sont expliqués étape par étape.


## Installation et importation des modules nécessaires

Avant de commencer à construire le système, tu dois d'abord importer quelques modules. Ceux-ci contiennent des fonctions préprogrammées dont tu auras besoin par la suite.


In [ ]:
# Installer les modules
!pip install opencv-contrib-python==4.10.0.84

In [ ]:
# Importer les modules
import matplotlib.pyplot as plt
from PIL import Image
from scripts import helpers
import numpy as np
from sklearn.model_selection import train_test_split

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, InputLayer, BatchNormalization

## Collecte des données

Les systèmes d'IA apprennent des règles à partir de données. La qualité d'un système d'IA dépend donc de la qualité de l'ensemble de données. Il existe plusieurs conditions auxquelles ton ensemble de données doit répondre pour être considéré comme de qualité.

* **Étiquettes correctes** : Les informations dans l'ensemble de données doivent être correctes. Par exemple, les images de chats doivent avoir l'étiquette 'chat', et celles des chiens l'étiquette 'chien'. Les images mal étiquetées perturberont le système d'IA.
* **Informations complètes** : L'ensemble de données doit contenir toutes les informations nécessaires pour résoudre le problème. Par exemple, si tu veux détecter des chats et des chiens, il est préférable d'avoir des photos de toutes les races de chats et de chiens.
* **Éléments uniques** : Les éléments de l'ensemble de données doivent être uniques. Chaque photo d'un chat ou d'un chien doit apparaître une seule fois dans l'ensemble de données. Les photos répétées n’aident pas à améliorer le système.
* **Équilibre** : Il y a autant d'exemples pour chaque type d'élément. Par exemple, autant d'exemples de chats que de chiens.
* **Éthique** : As-tu acquis l'ensemble de données de manière éthique ? Y a-t-il des droits d'auteur sur les données ? Les données contiennent-elles des informations personnelles ?

Dans cette activité, tu vas constituer toi-même un ensemble de données. Cela te permettra de veiller à sa qualité. Tu verras qu'il faut beaucoup de travail pour créer un ensemble de données. Constituer un ensemble de données de qualité est souvent l'un des principaux défis dans le développement d'un système d'IA.


### Dessiner des émotions

Tu vas construire un système capable de détecter des émotions. Cependant, tu ne commenceras pas immédiatement avec des photos de personnes, car cela nécessiterait un système trop complexe. Tu commenceras par détecter les émotions des émoticônes : tu tenteras de différencier les **émoticônes souriantes** des **émoticônes étonnées**.<br> Voici un exemple d'émoticône souriante et d'émoticône étonnée.

![](images/voorbeeld_blij.png)

![](images/voorbeeld_verbaasd.png)

Pour entraîner le système d'IA, tu auras besoin d'une centaine d'émoticônes souriantes et d'une centaine d'émoticônes étonnées. Oui, tu devras les dessiner toi-même et cela demandera un certain travail. Pas de souci, une méthode est prévue pour que tu puisses facilement le faire en utilisant un modèle. Le modèle contient une grille. Dans chaque case de la grille, tu dessineras une émoticône. **Sur chaque feuille, tu dessineras uniquement une émotion**, soit toutes des émoticônes souriantes, soit toutes des émoticônes étonnées.<br>Voici un exemple d'un modèle rempli.

![](images/voorbeeld_raster_blij.jpg)


**Tâche** : Imprime le document *raster.pdf* 10 fois au format A3 : 5 feuilles pour les émoticônes souriantes et 5 feuilles pour les émoticônes étonnées. Remplis la grille en dessinant des émoticônes. Astuce : tu peux répartir les feuilles entre plusieurs personnes, cela permettra de diviser le travail de dessin.


**Tâche** : Prends des photos des modèles remplis. Assure-toi que les quatre repères aux coins de la grille sont visibles sur la photo. Veille également à ce que la photo montre clairement les émoticônes et les repères.


## Charger les données

Maintenant que tu as collecté un ensemble de données, tu dois le charger dans Python.  
À gauche dans l'explorateur de fichiers, tu vois un dossier nommé *dataset*. À l'intérieur, il y a deux sous-dossiers nommés *blij* et *verbaasd*.

**Tâche** : Ajoute les photos des modèles dans le bon dossier. **ATTENTION ! Les images doivent avoir une extension .jpg, .png ou .jpeg !**

Pour télécharger un fichier dans ce dossier, clique sur l'icône de téléchargement en haut de l'explorateur de fichiers. L'icône de téléchargement est indiquée par une flèche verte sur l'image ci-dessous.

![](images/hoe_uploaden.png)


Il existe plusieurs fonctions prévues pour faciliter le traitement des données. Dans la cellule de code suivante, une fonction est appelée avec deux paramètres. Le premier paramètre est le dossier contenant les images des émoticônes souriantes, et le deuxième paramètre est l'étiquette des éléments dans ce dossier.


In [ ]:
# Charge toutes les photos du dossier 'dataset/blij' et attribue-leur l'étiquette 'blij'
# Les modèles avec les images seront automatiquement découpés en morceaux

images_heureuses, labels_heureuses = helpers.charge_fichiers_dans_dossier_avec_label("dataset/heureux", label="heureux")

Les images des émoticônes souriantes ainsi que leurs étiquettes sont référencées par des variables.

**Question** : Quelle variable fait référence aux images et laquelle fait référence aux étiquettes ?<br>
**Réponse** :


Maintenant que tu as chargé les images des émoticônes souriantes, regarde à quoi elles ressemblent. La cellule de code suivante contient le code pour afficher différentes propriétés de l'ensemble de données.


In [ ]:
print(f"Le dataset contient {len(images_heureuses)} images avec l'étiquette 'heureux'")
print(f"Les étiquettes sont : {labels_heureuses}")
print(f"La première image a une taille de {images_heureuses[0].shape}")
print("Les six premières images ressemblent à ceci :")
helpers.afficher_images(images_heureuses, labels_heureuses, max_afbeeldingen=6)

**Tâche** : Complète la cellule de code suivante pour faire référence aux images des émoticônes étonnées ainsi qu'à leur étiquette à l'aide d'une variable.


In [ ]:
# Complète le code aux endroits où ___ est indiqué
# Référence les images des émoticônes étonnées avec une variable
image_etonne, labels_etonne = helpers.charge_fichiers_dans_dossier_avec_label(___, label=___)

**Tâche** : Complète également la cellule de code suivante pour afficher les informations sur les émoticônes étonnées.


In [ ]:
# Complète le code aux endroits où ___ est indiqué
# Afficher les informations sur les émoticônes étonnées
print(f"Le jeu de données contient {___} images avec le label '___'")
print(f"Les étiquettes sont : {___}")
print(f"La première image a une taille de {___}")
print("Les six premières images ressemblent à ceci :")
helpers.afficher_images(___, ___, max_afbeeldingen=6)

## Préparer les données pour le système d'IA

Maintenant que tu as chargé les **images étiquetées** dans Python, tu dois les traiter dans un format que le système d'IA peut utiliser. Pour cela, tu dois suivre les étapes suivantes :
1. Fusionner les images souriantes et étonnées pour créer un seul ensemble de données.
2. Convertir les étiquettes de texte en nombres.
3. Diviser cet ensemble de données en trois sous-ensembles.
    * L'ensemble d'entraînement : il est utilisé pour entraîner le système d'IA.
    * L'ensemble de validation : il est utilisé pour valider les performances du système d'IA pendant son développement. Les images de cet ensemble ne chevauchent pas celles de l'ensemble d'entraînement. Cet ensemble est nécessaire pour vérifier si le système d'IA peut généraliser et ne s'est pas simplement mémorisé les images de l'ensemble d'entraînement.
    * L'ensemble de test : il est utilisé pour tester les performances du système d'IA après son développement. Les images de cet ensemble ne chevauchent pas celles des ensembles d'entraînement et de validation.

Peut-être n'est-ce pas encore tout à fait clair pourquoi ces ensembles sont nécessaires. Cela devrait devenir plus évident plus loin dans ce notebook. Dans les cellules suivantes, tu commenceras à préparer ces ensembles.


### Étape 1 : Fusionner les images et les étiquettes

À l'aide du code suivant, tu vas fusionner toutes les images dans un *tableau NumPy*. Tu feras de même pour toutes les étiquettes.


In [ ]:
images = np.vstack([np.array(images_heureuses), np.array(image_etonne)])
images = np.expand_dims(images, axis=-1)
labels = np.concatenate([np.array(labels_heureuses), np.array(labels_etonne)])

Afficher les informations sur les tableaux.


In [ ]:
print(f"Le jeu de données contient {images.shape[0]} images")
print(f"Il y a {len(labels)} étiquettes")
print(f"La première image a une taille de {images[0].shape}")

**Tâche** : Vérifie le nombre d'images dans l'ensemble de données. Est-ce que cela correspond à la somme du nombre d'images souriantes et du nombre d'images étonnées ?


### Étape 2 : Convertir les étiquettes de texte en nombres

Comme les ordinateurs peuvent effectuer des calculs plus rapidement et plus efficacement avec des nombres, tu vas convertir les étiquettes de texte en nombres via l'**encodage one-hot**. Chaque étiquette sera représentée par deux chiffres. Le premier chiffre est un 1 lorsque l'étiquette est *souriante* et un 0 lorsque l'étiquette est *étonnée*. Le deuxième chiffre est un 0 lorsque l'étiquette est *souriante* et un 1 lorsque l'étiquette est *étonnée*.<br>
Voici un exemple :

![](images/formaat_labels.png)


In [ ]:
# Encodage one-hot des étiquettes
labels_one_hot = helpers.one_hot_encode_labels(labels, ["heureux", "etonne"])

Maintenant que les étiquettes sont numériques, affiche 10 images aléatoires avec leur nouvelle étiquette.


In [ ]:
# Générer 10 indices aléatoires
random_indices = np.random.randint(0, len(labels), 10)
# Afficher ces 10 images aléatoires
helpers.afficher_images(images[random_indices], labels_one_hot[random_indices], max_afbeeldingen=10)

### Étape 3 : Diviser l'ensemble de données en ensembles d'entraînement, de test et de validation

Pour diviser l'ensemble de données en ces trois sous-ensembles, tu utiliseras la fonction *train_test_split()* du module *sklearn*. Tu l'utiliseras deux fois, d'abord pour séparer un ensemble de test, puis pour obtenir un ensemble d'entraînement et de validation.


#### Obtenir l'ensemble de test

En exécutant la cellule de code suivante, 20 % de l'ensemble de données seront sélectionnés au hasard comme ensemble de test.


In [ ]:
trainval_images, test_images, trainval_labels, test_labels = train_test_split(images, labels_one_hot, test_size=0.2)

Vérifie le format de l'ensemble de test et de l'ensemble des autres images qui seront utilisées pour développer le système.


In [ ]:
print(f"De ontwikkelingsdataset bevat {trainval_images.shape[0]} afbeeldingen")
print(f"De testverzameling bevat {test_images.shape[0]} afbeeldingen")

**Tâche** : Complète le code ci-dessous afin que les `trainval_images` et `trainval_labels` soient divisés en un ensemble d'entraînement et un ensemble de validation. 10 % de ces images doivent être utilisées comme ensemble de validation.

In [ ]:
# Complète le code aux endroits où ___ est indiqué
train_images, val_images, train_labels, val_labels = train_test_split(___, ___, test_size=___)

In [ ]:
print(f"La collection d'entraînement contient {___} images")
print(f"La collection de validation contient {___} images")

## Entraîner le système d'IA

Maintenant que l'ensemble de données est prêt, tu peux entraîner le système d'IA. Pour cela, tu utiliseras un réseau de neurones. Dans la cellule suivante, une fonction est définie pour spécifier la structure du réseau de neurones. La cellule de code ci-dessous fournit une représentation visuelle de ce réseau.

Il n'est pas nécessaire que tu comprennes déjà comment fonctionne un tel système d'IA. Tu peux considérer le réseau de neurones comme un système qui cherche les règles nécessaires pour reconnaître les émotions, ou en d'autres termes : le système *apprend* à reconnaître les émotions. Le réseau apprend ces règles en observant des exemples. Tu n'as pas besoin de comprendre les détails, mais sache qu'un tel réseau va transformer une image, étape par étape, en deux nombres. Ces nombres indiquent à quel point le système est sûr de sa décision. Le premier nombre représente la certitude du système que l'image contient une émotion joyeuse, le second combien il est certain qu'il s'agit d'une émotion surprise.<br>
Le système fera finalement une 'prédiction' pour une image. Il attribuera l'image à la classe dont il est le plus certain.


In [ ]:
# Définir la structure du modèle d'IA qui sera entraîné
def creer_reseau_neuronal(hauteur_image, largeur_image):
    model = Sequential()
    
    model.add(InputLayer(input_shape=(hauteur_image, largeur_image, 1)))
    
    model.add(Conv2D(1, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(2, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(4, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(8, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Conv2D(16, (3, 3), activation="relu"))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.1))
    
    model.add(Flatten())
    model.add(Dense(16, activation="relu"))   
    model.add(Dropout(0.1)) 
    model.add(Dense(2, activation="softmax"))
    
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    
    return model

![](images/neural_network_visualized.png)

L'instruction dans la cellule de code suivante appelle la fonction qui crée le modèle.

In [ ]:
model = creer_reseau_neuronal(images.shape[1], images.shape[2])

Avec la fonction `summary()`, tu peux afficher les détails du modèle.<br>Note que tu vois également combien de *paramètres* le système doit apprendre. Plus il y a de paramètres, plus le modèle est complexe et donc plus de données sont nécessaires pour l'entraîner.


In [ ]:
model.summary()

Maintenant, tu vas entraîner le modèle en utilisant l'ensemble de données. Lorsque tu exécutes la cellule suivante, tu verras que le réseau commence à apprendre à partir de l'ensemble d'entraînement. Tu obtiendras plusieurs informations :
* Tu verras quel *epoch* est en cours. Cela indique combien de fois l'ensemble d'entraînement complet a été présenté au réseau.
* L'*accuracy* est calculée en divisant le nombre de prédictions correctes par le nombre total de prédictions. Plus l'accuracy est élevée, meilleure est la performance du réseau sur l'ensemble d'entraînement.
* La *loss* indique dans quelle mesure les prédictions du réseau s'écartent en moyenne de la valeur correcte. Plus la loss est élevée, plus la performance du réseau sur l'ensemble d'entraînement est mauvaise.

À chaque epoch, l'*accuracy* et la *loss* sont également calculées pour l'ensemble de validation, afin de voir comment le système performe sur des données différentes de celles de l'entraînement.
* Plus la *val_accuracy* est élevée, meilleure est la performance du réseau sur l'ensemble de validation.
* Plus la *val_loss* est élevée, plus la performance du réseau sur l'ensemble de validation est mauvaise.

Ces valeurs sont utilisées pour ajuster l'entraînement du réseau.


In [ ]:
model.fit(train_images, train_labels, epochs=10, batch_size=1, validation_data=(val_images, val_labels))

Avec la cellule de code suivante, tu peux examiner plus en détail la *val_accuracy*.

In [ ]:
# accuracy op de validatieverzameling berekenen
loss, accuracy = model.evaluate(val_images, val_labels)
print(f"Précision de validation : {accuracy}")

La valeur que tu obtiendras ici sera toujours un peu différente. Elle dépend, entre autres, de la qualité de l'ensemble de données utilisé ici. Lors de nos tests, nous obtenons souvent une accuracy d'environ 0,95, soit 95 %.

L'*accuracy* te donne une idée des performances du système, mais elle ne nous indique pas à quel point il peut reconnaître correctement chaque groupe. Pour avoir plus de visibilité à ce sujet, tu peux examiner la *matrice de confusion*. Cette matrice indique combien d'images de chaque catégorie ont été correctement prédites. Ci-dessous, tu vois un exemple de matrice de confusion que nous avons obtenue pour notre modèle.

![](images/voorbeeld_confusion_matrix.png)

Ici, tu vois donc que 18 images joyeuses ont été correctement prédites. Deux des images joyeuses ont été prédites à tort, avec l'étiquette "surpris". Toutes les images surprises ont été correctement prédites.


Exécute le code suivant pour obtenir un aperçu de la matrice de confusion du modèle que tu as entraîné.

In [ ]:
# Imprimer la matrice de confusion de l'ensemble de validation
predictions = model.predict(val_images)
confusion_matrix = helpers.create_confusion_matrix_for_one_hot_encoded_labels(val_labels, predictions, ["heureux", "étonné"])
helpers.visualize_confussion_matrix_in_heatmap(confusion_matrix, ["heureux", "étonné"])

Tu peux également afficher une image de l'ensemble de validation avec la prédiction.

In [ ]:
mapped_labels_true = ["heureux" if np.argmax(label) == 0 else "étonné" for label in val_labels]
mapped_labels_predicted = ["heureux" if np.argmax(label) == 0 else "étonné" for label in predictions]
mapped_labels_combined = [f"Vrai: {mapped_labels_true[i]} \n La valeur prédite: {mapped_labels_predicted[i]}" for i in range(len(mapped_labels_true))]
helpers.afficher_images(val_images, mapped_labels_combined, max_afbeeldingen=len(val_images))

## Tester le système d'IA

Maintenant que tu as entraîné le système d'IA sur les ensembles d'entraînement et de validation, tu vas vérifier s'il fonctionne également sur des images qu'il n'a jamais vues auparavant. Cela permet de vérifier si le modèle a réellement appris à reconnaître les caractéristiques des émotions et n'a pas simplement mémorisé les exemples donnés. <br>
Pour ce faire, tu utilises l'ensemble de test. Celui-ci contient des images qui n'ont pas été utilisées pour entraîner le système d'IA. Si le système fonctionne sur ces images, cela signifie que le modèle a bien appris les caractéristiques des images émotionnelles. On dit dans ce cas que le modèle peut *généraliser* à d'autres exemples.

Pour mesurer la performance du système d'IA, il existe plusieurs mesures que tu peux effectuer. Une mesure simple est l'*accuracy*, que tu connais déjà pour les ensembles d'entraînement et de validation. L'*accuracy* est le rapport entre le nombre d'images correctement prédites et le nombre total de prédictions. Dans la cellule ci-dessous, tu calculeras l'*accuracy* sur l'ensemble de test.


In [ ]:
# Accuracy sur l'ensemble de test
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print(f"Test accuracy: {test_accuracy}")

# Matrice de confusion de l'ensemble de test
predictions = model.predict(test_images)
confusion_matrix = helpers.create_confusion_matrix_for_one_hot_encoded_labels(test_labels, predictions, ["heureux", "étonné"])
helpers.visualize_confussion_matrix_in_heatmap(confusion_matrix, ["heureux", "étonné"])

In [ ]:
# Images de l'ensemble de test avec leur prédiction
mapped_labels_true = ["heureux" if np.argmax(label) == 0 else "étonné" for label in test_labels]
mapped_labels_predicted = ["heureux" if np.argmax(label) == 0 else "étonné" for label in predictions]
mapped_labels_combined = [f"Vrai: {mapped_labels_true[i]} \n La valeur prédite: {mapped_labels_predicted[i]}" for i in range(len(mapped_labels_true))]
helpers.afficher_images(test_images, mapped_labels_combined, max_afbeeldingen=len(test_images))

Si le modèle ne fonctionne pas encore correctement, tu peux choisir d'ajuster la structure du modèle ou les paramètres d'entraînement.

Mais normalement, tu devrais déjà avoir un système qui fonctionne assez bien. Il n'est donc probablement pas nécessaire de modifier les paramètres du modèle. Si le modèle ne fonctionne pas encore correctement, tu peux choisir d'ajuster ce modèle pour obtenir de meilleurs résultats sur l'ensemble de validation. Pour t'assurer que tes ajustements fonctionnent bien sur d'autres données, teste-les également sur l'ensemble de test.

## Une image personnelle

Tu devrais maintenant avoir un système qui est relativement bon pour distinguer les smileys souriants des smileys surpris. Tu peux tester à nouveau le fonctionnement avec de nouveaux dessins. Imprime à nouveau le modèle avec les cases. Dessine dans les cases des smileys. Une partie des smileys est surprise, l'autre sourit. Après avoir dessiné, prends à nouveau une photo de ta feuille et charge-la dans le dossier *dataset/eigen*.

Lis les images comme tu l'as fait précédemment.

In [ ]:
# Charge toutes les images du dossier 'dataset/eigen' et attribue-leur le label 'inconnu'
# Les grilles sur ces images seront automatiquement découpées en morceaux
images_propres, _ = helpers.laadt_bestanden_in_map_met_label(___, label=___)
images_propres = np.expand_dims(images_propres, axis=-1)

Fais une prédiction sur la nouvelle image.


In [ ]:
predictions = model.predict(___)

Affiche le résultat de la prédiction.

In [ ]:
mapped_labels_predicted = ["blij" if np.argmax(label) == 0 else "verbaasd" for label in predictions]
helpers.afficher_images(images_propres, mapped_labels_predicted, max_afbeeldingen=len(images_propres))

# Défi

Tu as maintenant un système capable de distinguer deux émotions. Tu peux facilement étendre ce système à trois émotions. Pour cela, tu dois d'abord collecter des données supplémentaires. Une fois que tu as ces données, tu peux te baser sur le code ci-dessus pour créer un nouveau système capable de distinguer trois émotions.


# Extension : plus que des émotions

Tu peux en principe utiliser ce système pour distinguer n'importe quels deux objets. Tu pourrais donc, au lieu d'images de smileys, créer des dessins de chats et de chiens. Tu pourrais alors utiliser le même système pour distinguer ces dessins les uns des autres.


# Avec le soutien de 
![Vlaio](images/vlaio.png)